In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import math
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import OneHotEncoder

import sys
sys.path.append('../..')
# from mount_drive import mount_s_drive

In [ ]:
# mount_s_drive()

In [ ]:
myHours = 60*6

In [ ]:
database_folder = '/projects/LCICM/ACCMPMAP/'

In [ ]:
def read_by_chunks(file_path,chunk_size=1e6,head='infer'):
    df_chunks = []
    num_chunks = 0
    
    for chunk in pd.read_csv(file_path,chunksize=chunk_size,header=head):
        df_chunks.append(chunk)
        if num_chunks % 20 == 0:
            print(f'Chunk {num_chunks} Processed.')
        num_chunks += 1
        del chunk
    print('Processing finished.')
    
    return pd.concat(df_chunks, ignore_index=True)

### Patients 

In [ ]:
myOslerIds = pd.read_csv('./ohca_osler_ids.csv')
myPatEncIds = pd.read_csv('./ohca_pat_enc_ids.csv')
ohca_df = pd.read_csv('./ohca.csv')

### myPredictorsDf

In [ ]:
myPredictorsDf = ohca_df[['osler_id', 'gender', 'birth_date', 'dx_name', 'hosp_admsn_time', 'disch_disp_c', 'ed_visit_yn']]
myPredictorsDf['gender'] = (myPredictorsDf['gender'] == 'Male').astype(int)
myPredictorsDf['ed_visit_yn'] = (myPredictorsDf['ed_visit_yn'] == 'Y').astype(int)
myPredictorsDf['birth_date'] = pd.to_datetime(myPredictorsDf['birth_date'],errors='coerce')
myPredictorsDf['hosp_admsn_time'] = pd.to_datetime(myPredictorsDf['hosp_admsn_time'],errors='coerce')
myPredictorsDf['age'] = (myPredictorsDf['hosp_admsn_time']-myPredictorsDf['birth_date']).apply(lambda x: math.floor(x.days/365.25))
myPredictorsDf = myPredictorsDf.drop(columns=['hosp_admsn_time'])
myPredictorsDf.loc[myPredictorsDf['age'] > 100, 'age'] = 100
myPredictorsDf = myPredictorsDf.drop(columns=['birth_date'])
patient_ids = myPredictorsDf.pop('osler_id')
myPredictorsDf.insert(0, 'osler_id', patient_ids)
myPredictorsDf['death_at_disch'] = (myPredictorsDf['disch_disp_c'] == 20.0).astype(int)
myPredictorsDf = myPredictorsDf.drop(columns=['disch_disp_c'])
myPredictorsDf = myPredictorsDf.reset_index(drop=True)

In [ ]:
#myPredictorsDf.head()

### Extraction Functions

In [ ]:
def getFeaturesFromDf(aDf, aTimeColumn, aTypeColumn, aValueColumn):
    aDf['num_values'] = pd.to_numeric(aDf[aValueColumn], errors='coerce')
    myMasterGroupDf = aDf[(aDf[aTimeColumn] < myHours)].groupby(['osler_id', aTypeColumn])
    myMasterGroupBegin = aDf.loc[myMasterGroupDf[aTimeColumn].idxmin()]
    myMasterGroupEnd = aDf.loc[myMasterGroupDf[aTimeColumn].idxmax()]
    myMasterGroupAgg = myMasterGroupDf.agg({'mean', 'min', 'max'})
    myMasterGroupAgg = myMasterGroupAgg.num_values.reset_index()
    return myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg

In [ ]:
def mergeFeaturesInDf(aDfToMerge, aBegin, aEnd, aAgg, aPrefix, aTypeColumn, aValueColumn):
    osler_ids = aDfToMerge[['osler_id']].copy()
    myPredictorsDf1 = aDfToMerge.copy()
    
    print('Running Begin Columns')
    total = int(aBegin[aTypeColumn].nunique() / 10)
    begin_cols = {}
    i = 0
    print('progress: ', end="")
    for value in aBegin[aTypeColumn].unique():
        myDf = aBegin[aBegin[aTypeColumn] == value]
        merged = osler_ids.merge(myDf[['osler_id', aValueColumn]], on='osler_id', how='left')
        begin_cols[aPrefix + '_first_' + value] = merged[aValueColumn]
        if (i % total == 0):
            print('#', end="")
        i+= 1
    begin_df = pd.concat([osler_ids, pd.DataFrame(begin_cols)], axis=1)
    print()

    print('Running End Columns')
    total = int(aEnd[aTypeColumn].nunique() / 10)
    end_cols = {}
    i = 0
    print('progress: ', end="")
    for value in aEnd[aTypeColumn].unique():
        myDf = aEnd[aEnd[aTypeColumn] == value]
        merged = osler_ids.merge(myDf[['osler_id', aValueColumn]], on='osler_id', how='left')
        end_cols[aPrefix + '_last_' + value] = merged[aValueColumn]
        if (i % total == 0):
            print('#', end="")
        i+= 1
    end_df = pd.concat([osler_ids, pd.DataFrame(end_cols)], axis=1)
    print()
    
    print('Running Agg Columns')
    total = int(aAgg[aTypeColumn].nunique() / 10)
    agg_cols = {}
    i = 0
    print('progress: ', end="")
    for value in aAgg[aTypeColumn].unique():
        myDf = aAgg[aAgg[aTypeColumn] == value]
        merged = osler_ids.merge(myDf[['osler_id', 'max', 'min', 'mean']], on='osler_id', how='left')
        agg_cols.update({aPrefix + '_max_' + value: merged['max'],
                         aPrefix + '_min_' + value: merged['min'],
                         aPrefix + '_mean_' + value: merged['mean']})
        if (i % total == 0):
            print('#', end="")
        i+= 1
    agg_df = pd.concat([osler_ids, pd.DataFrame(agg_cols)], axis=1)
    print()

    myPredictorsDf1 = myPredictorsDf1.merge(begin_df, on='osler_id', how='left')
    myPredictorsDf1 = myPredictorsDf1.merge(end_df, on='osler_id', how='left')
    myPredictorsDf1 = myPredictorsDf1.merge(agg_df, on='osler_id', how='left')
    
    return myPredictorsDf1

### Flowsheets

In [ ]:
def read_flowsheet(file_path,chunk_size=1e6,head='infer'):
    df_chunks = []
    num_chunks = 0    
    for chunk in pd.read_csv(file_path,chunksize=chunk_size,header=head):
        chunk.columns = ['osler_id','pat_enc_csn_id','inpatient_data_id','recorded_time','meas_value',
                         'meas_comment','meas_id','meas_row_type_c','meas_val_type_c','meas_template_id',
                         'meas_fsd_id','meas_line','meas_occurance','rw_meas_id',
                         'ip_lda_id','rw_row_type_c','rw_val_type_c','meas_taken_user_id']
        chunk = chunk[chunk['osler_id'].isin(myOslerIds['osler_id'])]
        df_chunks.append(chunk)
        if num_chunks % 20 == 0:
            print(f'Chunk {num_chunks} Processed.')
        num_chunks += 1
        del chunk
    print('Processing finished.')    
    return pd.concat(df_chunks, ignore_index=True)

In [ ]:
flowsheet_df1 = read_flowsheet(database_folder+'Isalis/Isalis_Flowsheet_part1.csv',head=None)
flowsheet_df2 = read_flowsheet(database_folder+'Isalis/Isalis_Flowsheet_part2.csv',head=None)
flowsheet_df3 = read_flowsheet(database_folder+'Isalis/Isalis_Flowsheetdata.csv',head=None)
flowsheet_df = pd.concat([flowsheet_df1,flowsheet_df2,flowsheet_df3],ignore_index=True)

In [ ]:
flowsheet_df = pd.merge(flowsheet_df,ohca_df[['osler_id','hosp_admsn_time','hosp_disch_time']],on='osler_id',how='right')
flowsheet_df['recorded_time'] = pd.to_datetime(flowsheet_df['recorded_time'],errors='coerce')
flowsheet_df['hosp_admsn_time'] = pd.to_datetime(flowsheet_df['hosp_admsn_time'],errors='coerce')
flowsheet_df['hosp_disch_time'] = pd.to_datetime(flowsheet_df['hosp_disch_time'],errors='coerce')
flowsheet_df['meas_offset'] = (flowsheet_df['recorded_time']-flowsheet_df['hosp_admsn_time']).dt.total_seconds()/60
flowsheet_df = flowsheet_df[flowsheet_df['meas_offset']>=0]
flowsheet_df = flowsheet_df[flowsheet_df['recorded_time']<=flowsheet_df['hosp_disch_time']]
measures_df = pd.read_csv(database_folder+'Isalis/d_flo_measures.csv')
measures_df = measures_df.rename(columns={'flo_meas_id':'meas_id','flo_meas_name':'meas_name'})
measures_df['meas_name'] = measures_df['meas_name'].str.lower().str.replace(' ', '_')
flowsheet_df = flowsheet_df.sort_values(by=['osler_id','meas_offset'])
flowsheet_df = flowsheet_df.merge(measures_df[['meas_id','meas_name']], on='meas_id', how='left')

In [ ]:
columns_to_keep = ['osler_id','meas_value','meas_offset','meas_name','meas_id','meas_val_type_c']
flowsheet_df_tmp = flowsheet_df[columns_to_keep]
flowsheet_df_tmp = flowsheet_df_tmp.dropna(subset='meas_name')
flowsheet_df_tmp = flowsheet_df_tmp[flowsheet_df_tmp['meas_offset']<=myHours]

#### Num + BP + Height + Weight + Temp

In [ ]:
# num_flowsheet_df = flowsheet_df_tmp[flowsheet_df_tmp['meas_val_type_c']==1]
# height_flowsheet_df = flowsheet_df_tmp[flowsheet_df_tmp['meas_id']==11]
# weight_flowsheet_df = flowsheet_df_tmp[flowsheet_df_tmp['meas_id']==14]
# temp_flowsheet_df = flowsheet_df_tmp[flowsheet_df_tmp['meas_val_type_c']==7]
# bp_flowsheet_df = flowsheet_df_tmp[flowsheet_df_tmp['meas_val_type_c']==4]
# bp_flowsheet_df[['bp_systolic', 'bp_diastolic']] = bp_flowsheet_df['meas_value'].str.split('/', expand=True)
# bp_flowsheet_df = bp_flowsheet_df.melt(id_vars=['osler_id','meas_offset','meas_id'], value_vars=['bp_systolic','bp_diastolic'], var_name='meas_name', value_name='meas_value')
# ed_acuity_df = flowsheet_df_tmp[flowsheet_df_tmp['meas_id']==16054]
# merged_df = pd.concat([num_flowsheet_df, bp_flowsheet_df, height_flowsheet_df, weight_flowsheet_df, temp_flowsheet_df, ed_acuity_df])
# merged_df['meas_value'] = pd.to_numeric(merged_df['meas_value'], errors='coerce')
# merged_df = merged_df.dropna(subset=['meas_value'])
# merged_df['meas_id'].nunique()

import pandas as pd

# --- subsets (always .copy() when you’ll modify) ---
num_flowsheet_df    = flowsheet_df_tmp[flowsheet_df_tmp['meas_val_type_c'] == 1].copy()
height_flowsheet_df = flowsheet_df_tmp[flowsheet_df_tmp['meas_id'] == 11].copy()
weight_flowsheet_df = flowsheet_df_tmp[flowsheet_df_tmp['meas_id'] == 14].copy()
temp_flowsheet_df   = flowsheet_df_tmp[flowsheet_df_tmp['meas_val_type_c'] == 7].copy()
ed_acuity_df        = flowsheet_df_tmp[flowsheet_df_tmp['meas_id'] == 16054].copy()

# --- BP: split systolic/diastolic and reshape long ---
bp_flowsheet_df = flowsheet_df_tmp[flowsheet_df_tmp['meas_val_type_c'] == 4].copy()

# extract systolic/diastolic as digits (handles "120/80", "120 / 80 mmHg", etc.)
bp_parts = bp_flowsheet_df['meas_value'].astype(str).str.extract(
    r'(?P<bp_systolic>\d+\.?\d*)\s*/\s*(?P<bp_diastolic>\d+\.?\d*)'
)
bp_flowsheet_df[['bp_systolic', 'bp_diastolic']] = bp_parts

# melt WITHOUT naming conflict (since meas_value already exists)
bp_long = bp_flowsheet_df.melt(
    id_vars=['osler_id', 'meas_offset', 'meas_id'],
    value_vars=['bp_systolic', 'bp_diastolic'],
    var_name='meas_name',
    value_name='bp_value'
)

# put into the same column name used everywhere else
bp_long = bp_long.rename(columns={'bp_value': 'meas_value'})

# optional: make meas_name prettier / consistent
bp_long['meas_name'] = bp_long['meas_name'].map({
    'bp_systolic': 'bp_systolic',
    'bp_diastolic': 'bp_diastolic'
})

# --- merge ---
merged_df = pd.concat(
    [num_flowsheet_df, bp_long, height_flowsheet_df, weight_flowsheet_df, temp_flowsheet_df, ed_acuity_df],
    ignore_index=True
)

# --- coerce numeric and drop missing ---
merged_df['meas_value'] = pd.to_numeric(merged_df['meas_value'], errors='coerce')
merged_df = merged_df.dropna(subset=['meas_value'])

# result
merged_df['meas_id'].nunique()


In [ ]:
myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg = getFeaturesFromDf(merged_df, 'meas_offset', 'meas_name', 'meas_value')
myPredictorsDf1 = mergeFeaturesInDf(myPredictorsDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg, 'flo', 'meas_name', 'meas_value')

#### Str + Custom

In [ ]:
str_flowsheet_df = flowsheet_df_tmp[flowsheet_df_tmp['meas_val_type_c']==2]
str_meas_counts = str_flowsheet_df.groupby('meas_id')['osler_id'].nunique()
str_meas_ids = str_meas_counts[str_meas_counts >= 100].index
str_measures_df = measures_df[measures_df['meas_id'].isin(str_meas_ids)]
str_flowsheet_df = str_flowsheet_df[str_flowsheet_df['meas_id'].isin(str_meas_ids)]

In [ ]:
custom_flowsheet_df = flowsheet_df_tmp[flowsheet_df_tmp['meas_val_type_c']==8]
custom_meas_counts = custom_flowsheet_df.groupby('meas_id')['osler_id'].nunique()
custom_meas_ids = custom_meas_counts[custom_meas_counts >= 100].index
custom_measures_df = measures_df[measures_df['meas_id'].isin(custom_meas_ids)]
custom_flowsheet_df = custom_flowsheet_df[custom_flowsheet_df['meas_id'].isin(custom_meas_ids)]

In [ ]:
merged_df2 = pd.concat([str_flowsheet_df, custom_flowsheet_df])
print(merged_df2['meas_id'].nunique())
merged_df2 = merged_df2.dropna(subset=['meas_value'])
merged_df2['meas_value'] = merged_df2['meas_value'].str.split(';')
merged_df2 = merged_df2.explode('meas_value')

In [ ]:
encoder = OneHotEncoder(sparse_output=False, dtype=int)
encoded_matrix = encoder.fit_transform(merged_df2[['meas_value']])
encoded_df = pd.DataFrame(encoded_matrix, columns=encoder.get_feature_names_out(['meas_value']))
encoded_df['osler_id'] = merged_df2['osler_id'].values
encoded_df['meas_name'] = merged_df2['meas_name'].values
new_column_names = {old_col: f"flo_{meas_name}_{old_col.split('_')[-1]}" for old_col, meas_name in zip(encoded_df.columns[:-2], encoded_df['meas_name'])}
encoded_df.rename(columns=new_column_names, inplace=True)
bin_df = encoded_df.groupby('osler_id').max().reset_index()
bin_df = bin_df.drop(columns=['meas_name'])

In [ ]:
myPredictorsDf2 = myPredictorsDf.merge(bin_df, on='osler_id', how='left')
myPredictorsDf2 = myPredictorsDf2.fillna(0)

### Labs

In [ ]:
pip install pyarrow

In [ ]:
df_chunks = []
num_chunks = 0    
for chunk in pd.read_csv(database_folder+'/2024-08-15/dbo.accm_labs.csv',chunksize=10e6, on_bad_lines="skip"):
    chunk = pd.merge(chunk,ohca_df[['osler_id','hosp_admsn_time','hosp_disch_time']],on='osler_id',how='right')
    chunk['specimen_taken_time'] = pd.to_datetime(chunk['specimen_taken_time'],errors='coerce')
    chunk['hosp_admsn_time'] = pd.to_datetime(chunk['hosp_admsn_time'],errors='coerce')
    chunk['hosp_disch_time'] = pd.to_datetime(chunk['hosp_disch_time'],errors='coerce')
    chunk = chunk[chunk['specimen_taken_time']<=chunk['hosp_disch_time']]
    chunk['meas_offset'] = (chunk['specimen_taken_time']-chunk['hosp_admsn_time']).dt.total_seconds()/60
    chunk = chunk[(chunk['meas_offset']>=0)&(chunk['meas_offset']<=myHours)]
    df_chunks.append(chunk)
    if num_chunks % 20 == 0:
        print(f'Chunk {num_chunks} Processed.')
    num_chunks += 1
    del chunk
print('Processing finished.')    
labs_df = pd.concat(df_chunks, ignore_index=True)

In [ ]:
labs_df_tmp = labs_df.copy()
labs_df_tmp = labs_df_tmp[labs_df_tmp['data_type']=='Number']
proc_df = pd.read_csv(database_folder+'PMAP_commons_dbo_CLARITY_EAP.csv')
proc_df = proc_df.rename(columns={'PROC_ID':'proc_id','PROC_NAME':'proc_name'})
proc_df['proc_name'] = proc_df['proc_name'].str.lower().str.replace(' ', '_')
labs_df_tmp = labs_df_tmp.sort_values(by=['osler_id','meas_offset'])
labs_df_tmp = labs_df_tmp.merge(proc_df[['proc_id','proc_name']], on='proc_id', how='left')
columns_to_keep = ['osler_id','ord_value','meas_offset','proc_name','proc_id']
labs_df_tmp = labs_df_tmp[columns_to_keep]
labs_df_tmp = labs_df_tmp.dropna(subset='proc_name')
labs_df_tmp['ord_value'] = pd.to_numeric(labs_df_tmp['ord_value'], errors='coerce')
labs_df_tmp = labs_df_tmp.dropna(subset=['ord_value'])
labs_df_tmp['proc_id'].nunique()

In [ ]:
myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg = getFeaturesFromDf(labs_df_tmp, 'meas_offset', 'proc_name', 'ord_value')
myPredictorsDf3 = mergeFeaturesInDf(myPredictorsDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg, 'lab', 'proc_name', 'ord_value')

### Medication administrations

In [ ]:
df_chunks = []
num_chunks = 0   
for chunk in pd.read_csv(database_folder+'/2024-08-15/dbo.accm_med_admin.csv',chunksize=1e6):
    chunk = pd.merge(chunk,ohca_df[['osler_id','hosp_admsn_time','hosp_disch_time']],on=['osler_id','hosp_admsn_time','hosp_disch_time'],how='right')
    chunk['taken_time'] = pd.to_datetime(chunk['taken_time'],errors='coerce')
    chunk['hosp_admsn_time'] = pd.to_datetime(chunk['hosp_admsn_time'],errors='coerce')
    chunk['hosp_disch_time'] = pd.to_datetime(chunk['hosp_disch_time'],errors='coerce')
    chunk = chunk[chunk['taken_time']<=chunk['hosp_disch_time']]
    chunk['meas_offset'] = (chunk['taken_time']-chunk['hosp_admsn_time']).dt.total_seconds()/60
    chunk = chunk[(chunk['meas_offset']>=0)&(chunk['meas_offset']<=myHours)]
    df_chunks.append(chunk)
    if num_chunks % 20 == 0:
        print(f'Chunk {num_chunks} Processed.')
    num_chunks += 1
    del chunk
print('Processing finished.')
med_admin_df = pd.concat(df_chunks, ignore_index=True)

In [ ]:
meds_df = med_admin_df.copy()
meds_df['generic_name'] = meds_df['generic_name'].fillna(meds_df['medication_name'])
meds_df['generic_name'] = meds_df['generic_name'].str.split('(').str[0].str.strip()
meds_df['generic_name'] = meds_df['generic_name'].str.lower().str.replace(' ', '_', regex=True)
meds_df['generic_name'] = meds_df['generic_name'].str.replace(r'^zzz+|^zz+', '', regex=True)
meds_df['generic_name'].nunique()

In [ ]:
encoder = OneHotEncoder(sparse=False, dtype=int)
encoded_matrix = encoder.fit_transform(meds_df[['generic_name']])
encoded_df = pd.DataFrame(encoded_matrix, columns=encoder.get_feature_names_out(['generic_name']))
encoded_df.rename(columns=lambda col: col.replace('generic_name_', 'med_'), inplace=True)
encoded_df['osler_id'] = meds_df['osler_id'].values
bin_meds_df = encoded_df.groupby('osler_id').max().reset_index()

In [ ]:
myPredictorsDf4 = myPredictorsDf.merge(bin_meds_df,on='osler_id',how='left')
myPredictorsDf4 = myPredictorsDf4.fillna(0)

### Other Diagnoses

In [ ]:
# def nameSearchCardiacArrest(text):
#     text = str(text).lower()
#     patterns = [r'cardia.*rrest',
#                 r'cardio.*rrest',
#                 r'circulat.*rrest',
#                 r'\basystole',
#                 r'\basystolia',
#                 r'\bpea\b|pulseless elec.*act.*']
#     if any(re.search(pattern, text) for pattern in patterns):
#         exclusion_patterns = [r'history|hx|h/o',
#                               r'before cardiac arrest',
#                               r'due to cardiac arrest',
#                               r'neonatal',
#                               r'newborn']       
#         if not any(re.search(pattern, text) for pattern in exclusion_patterns):
#             return True   
#     return False

# def icdSearchCardiacArrest(text):
#     text = str(text).lower()
#     icd10_match = re.search(r'\bi46\.\b', text)
#     icd9_match = re.search(r'\b427\.5\b', text)
#     return bool(icd10_match or icd9_match)

In [ ]:
# encounter_dx_df = read_by_chunks(database_folder+'Databases/ACCMPMAP/2024-08-15/dbo.accm_encounter_dx.csv')

In [ ]:
# encounter_dx_df1 = encounter_dx_df[encounter_dx_df['pat_enc_csn_id'].isin(myPatEncIds['pat_enc_csn_id'])]
# encounter_dx_df1['name_search'] = encounter_dx_df1['dx_name'].transform(nameSearchCardiacArrest)
# encounter_dx_df1['icd9_icd10'] = encounter_dx_df1['icd9_code'] + ' ' + encounter_dx_df1['icd10_code']
# encounter_dx_df1['icd_search'] = encounter_dx_df1['icd9_icd10'].transform(icdSearchCardiacArrest)

In [ ]:
# dx_df = encounter_dx_df1[(~encounter_dx_df1['name_search'])&(~encounter_dx_df1['icd_search'])]
# dx_df['dx_name'] = dx_df['dx_name'].str.lower().str.replace(' ', '_', regex=True)

In [ ]:
# encoder = OneHotEncoder(sparse=False, dtype=int)
# encoded_matrix = encoder.fit_transform(dx_df[['dx_name']])
# encoded_df = pd.DataFrame(encoded_matrix, columns=encoder.get_feature_names_out(['dx_name']))
# encoded_df.rename(columns=lambda col: col.replace('dx_name_', 'dx_'), inplace=True)
# encoded_df['osler_id'] = dx_df['osler_id'].values
# bin_dx_df = encoded_df.groupby('osler_id').max().reset_index()

In [ ]:
# myPredictorsDf5 = myPredictorsDf.merge(bin_dx_df,on='osler_id',how='left')
# myPredictorsDf5 = myPredictorsDf5.fillna(0)

### Merge

In [ ]:
columns = ['osler_id', 'gender', 'dx_name', 'ed_visit_yn', 'age', 'death_at_disch']
myPredictorsDf = myPredictorsDf1
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf2, on=columns, how='left')
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf3, on=columns, how='left')
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf4, on=columns, how='left')
# myPredictorsDf = myPredictorsDf.merge(myPredictorsDf5, on=columns, how='left')

### CA Diagnosis

In [ ]:
myPredictorsDf['dx'] = myPredictorsDf['dx_name'].str.lower().str.split(r'; |- ')
mlb = MultiLabelBinarizer()
one_hot = pd.DataFrame(mlb.fit_transform(myPredictorsDf['dx']), columns=mlb.classes_)
df_encoded = myPredictorsDf[['osler_id']].join(one_hot)
myPredictorsDf = myPredictorsDf.merge(df_encoded, on='osler_id', how='left')
myPredictorsDf = myPredictorsDf.drop(columns=['dx_name', 'dx'])
myPredictorsDf['asystole'] = myPredictorsDf[['asystole', 'asystole by electrocardiogram', 'cardiac asystole']].max(axis=1, skipna=True)
myPredictorsDf['pea'] = myPredictorsDf[['pea (pulseless electrical activity)', 'pulseless electrical activity','cardiac arrest with pulseless electrical activity']].max(axis=1, skipna=True)
myPredictorsDf['cardiopulmonary arrest w/ resuscitation'] = myPredictorsDf[['cardiopulmonary arrest with successful resusciation',
                                                                            'cardiopulmonary arrest with successful resuscitation']].max(axis=1, skipna=True)
myPredictorsDf['VF'] = myPredictorsDf[['cardiac arrest with ventricular fibrillation', 'ventricular fibrillation']].max(axis=1, skipna=True)
myPredictorsDf.drop(columns=['cardiac arrest', 'cardiac arrest ', 'cardiac arrest, cause unspecified',
                             'asystole by electrocardiogram', 'cardiac asystole', 'pea (pulseless electrical activity)',
                             'pulseless electrical activity','cardiac arrest with pulseless electrical activity',
                             'cardiopulmonary arrest with successful resusciation', 'cardiopulmonary arrest with successful resuscitation',
                             'cardiac arrest with ventricular fibrillation', 'ventricular fibrillation'], inplace=True)

### GCS

In [ ]:
gcs_df = flowsheet_df[(flowsheet_df['meas_id']==30405069)|(flowsheet_df['meas_id']==160302)]
gcs_df['num_meas_value'] = pd.to_numeric(gcs_df['meas_value'], errors='coerce')
gcs_df = gcs_df.dropna(subset=['num_meas_value'])

In [ ]:
myGcsDf = gcs_df.sort_values(by=['osler_id', 'meas_offset'])
myGcsDf['num_meas_value'] = myGcsDf.groupby('osler_id')['num_meas_value'].ffill().bfill()
myGcsDf['meas_offset2'] = myGcsDf.meas_offset
myGcsDf2 = myGcsDf.groupby('osler_id').agg({'meas_offset':'min', 'meas_offset2': 'max'})
myGcsDfMin = myGcsDf.merge(myGcsDf2, on=['osler_id', 'meas_offset'], how='inner')
myGcsDfMax = myGcsDf.merge(myGcsDf2, on=['osler_id', 'meas_offset2'], how='inner')
myGcsDfMin['num_meas_value'] = myGcsDfMin['num_meas_value'].astype(int)
myGcsDfMax['num_meas_value'] = myGcsDfMax['num_meas_value'].astype(int)

In [ ]:
myPredictorsDf = myPredictorsDf.merge(myGcsDfMin[['osler_id', 'meas_offset', 'num_meas_value']], on='osler_id')
myPredictorsDf.rename(columns={'meas_offset': 'first_mGCS_time', 'num_meas_value': 'first_mGCS'}, inplace=True)
myPredictorsDf = myPredictorsDf.merge(myGcsDfMax[['osler_id', 'meas_offset2', 'num_meas_value']], on='osler_id')
myPredictorsDf.rename(columns={'meas_offset2': 'last_mGCS_time', 'num_meas_value': 'last_mGCS'}, inplace=True)

### Hypothermia

In [ ]:
temp_df = flowsheet_df[['osler_id','meas_value','meas_offset','meas_id']]
temp_df = temp_df[(temp_df['meas_id']==6)|(temp_df['meas_id']==304301490)]
temp_df = temp_df[temp_df['meas_offset']<=1440]
temp_df['num_meas_value'] = pd.to_numeric(temp_df['meas_value'], errors='coerce')
temp_df = temp_df.sort_values(by=['osler_id','meas_offset'])
temp_df['min_value'] = temp_df.groupby('osler_id')['num_meas_value'].transform('min')
temp_df = temp_df[temp_df['min_value']>=77]
grouped = temp_df.groupby('osler_id')
temp_df['time_diff'] = grouped['meas_offset'].diff()
temp_df['temp_x_time'] = temp_df['time_diff'] * temp_df['num_meas_value'].shift(1)
patient_avg = grouped.apply(lambda x: np.sum(x['temp_x_time']) / np.sum(x['time_diff']))
hyp_df = temp_df[temp_df['osler_id'].isin(patient_avg[patient_avg<96.8].index)]

In [ ]:
hyp_ids = hyp_df['osler_id'].drop_duplicates()
hyp_ids.to_csv('hyp_ids.txt',index=False)

In [ ]:
myPredictorsDf['hypothermia'] = myPredictorsDf['osler_id'].isin(hyp_ids).astype(int)

In [ ]:
#myPredictorsDf.head()

### Save

In [ ]:
#myPredictorsDf.to_csv('PMAP_Predictors.csv', index=False)

In [ ]:
myPredictorsDf.to_csv('PMAP_Predictors2.csv', index=False)